In [4]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm_notebook
pd.set_option("display.max_columns", 500)

EVENTS_DIR = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_enriched"
HOURLY_DIR = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_hourly_data"
VAULTS_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/common/vaults_list.csv"
SPIKES_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/all_spikes_dataset.csv"
SUPPLIERS_SHARE_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_suppliers_share.csv"
BORROWERS_SHARE_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_borrowers_share.csv"



CRYPTO_MARKETS = [
    'eth_wbtc_usdc', 'base_wbtc_usdt', 'eth_cbbtc_usdc',
    'eth_wbtc_usdt', "eth_wsteth_usdt", 'eth_weth_usdt', 'eth_cbbtc_usdt',

    'base_cbbtc_usdc_full',
    # 'base_wbtc_usdc',
    # 'base_wbtc_usdt',
]
PT_MARKETS = [
    "eth_PT-RLP-4SEP2025_usdc", "eth_PT-USD0++-27MAR2025_usdc",
    "eth_PT-USD0++-31OCT2024_usdc", "eth_PT-USDe-25SEP2025_dai",
    "eth_PT-USDe-25SEP2025_usdc", "eth_PT-USDe-25SEP2025_usdt",
    "eth_PT-USDe-27MAR2025_dai", "eth_PT-USDe-27NOV2025_usds",
    "eth_PT-USDe-31JUL2025_dai", "eth_PT-USR-29MAY2025_usdc",
    "eth_PT-csUSDL-31JUL2025_usdc", "eth_PT-lvlUSD-29MAY2025_usdc",
    "eth_PT-mHYPER-20NOV2025_usdc", "eth_PT-reUSD-18DEC2025_usdc",
    "eth_PT-reUSD-25JUN2026_usdc", "eth_PT-sNUSD-5MAR2026_usdc",
    "eth_PT-sdeUSD-1753142406_usdc", "eth_PT-slvlUSD-25SEP2025_usdc",
    "eth_PT-slvlUSD-29MAY2025_usdc", "eth_PT-stcUSD-23JUL2026_usdc",
    "eth_PT-stcUSD-29JAN2026_usdc", "eth_PT-syrupUSDC-28AUG2025_usdc",
    "eth_PT-syrupUSDC-30OCT2025_usdc", "eth_PT-wstUSR-25SEP2025_usdc",
    "eth_PT-wstUSR-27MAR2025_usdc", "eth_PT-wstUSR-27MAR2025_usr",
    "PT-reUSD-25JUN2026_usdc", "PT-siUSD-26MAR2026_usdc",
]
YB_TOKENS = [
    'eth_usr_usdc', 'eth_wsteth_usdc', 'eth_rlp_usdc',
    'eth_usd0++_usdc', 'eth_fxsave_usdc', 'eth_mapollo_usdc',
    'eth_wsrusd_usdc', 'eth_syrupusdc_pyusd', 'eth_susde_pyusd',
    'eth_stcusd_usdc', 'eth_usde_dai', 'eth_mhyper_usdc', 'eth_syrupusdc_usdc',
    'eth_wstusr_usdc','eth_slvlusd_usdc','eth_csusdl_usdc', 'eth_mF-ONE_usdc', 'eth_reusd_usdc',
    'eth_siusd_usdc', 'eth_sdeusd_usdc'
]


def load_market_events(market_name):
    file_path = os.path.join(EVENTS_DIR, f"{market_name}.csv")
    df = pd.read_csv(file_path)
    df['datetime'] = pd.to_datetime(df['datetime'])
    return df

def load_market_hourly(market_name):
    file_path = os.path.join(HOURLY_DIR, f"{market_name}.csv")
    df = pd.read_csv(file_path)
    df['datetime'] = pd.to_datetime(df['datetime'])
    return df

def load_spikes():
    df = pd.read_csv(SPIKES_PATH)
    df['spike_trigger_datetime'] = pd.to_datetime(df['spike_trigger_datetime'])
    df['spike_recovery_datetime'] = pd.to_datetime(df['spike_recovery_datetime'])
    return df

def load_suppliers_share():
    df = pd.read_csv(SUPPLIERS_SHARE_PATH)
    df['datetime'] = pd.to_datetime(df['datetime'])
    return df

def load_borrowers_share():
    df = pd.read_csv(BORROWERS_SHARE_PATH)
    df['datetime'] = pd.to_datetime(df['datetime'])
    return df

def get_closest_value(df, timestamp, column):
    idx = df['timestamp'].searchsorted(timestamp, side='right') - 1
    if idx < 0:
        return np.nan
    return df.iloc[idx][column]

def get_share_at_timestamp(share_df, market, timestamp, side):
    sub = share_df[(share_df['market'] == market) & (share_df['side'] == side)]
    if sub.empty:
        return None
    idx = sub['timestamp'].searchsorted(timestamp, side='right') - 1
    if idx < 0:
        return None
    return sub.iloc[idx]

def extract_user_positions(market_events):
    market_events = market_events.sort_values(['user_address', 'timestamp']).copy()
    positions = []
    
    for user, user_df in market_events.groupby('user_address'):
        user_df = user_df.sort_values('timestamp').reset_index(drop=True)
        open_indices = user_df[user_df['event_sequence_type'] == 'position_open'].index.tolist()
        close_indices = user_df[user_df['event_sequence_type'] == 'position_close'].index.tolist()
        
        for open_idx in open_indices:
            open_row = user_df.loc[open_idx]
            open_ts = open_row['timestamp']
            
            close_candidates = [c for c in close_indices if c > open_idx]
            if close_candidates:
                close_idx = close_candidates[0]
                close_row = user_df.loc[close_idx]
                close_ts = close_row['timestamp']
                position_events = user_df.loc[open_idx:close_idx]
                is_closed = True
            else:
                close_row = None
                close_ts = None
                position_events = user_df.loc[open_idx:]
                is_closed = False
            
            open_borrow_rate = open_row['borrow_rate_after']
            
            if is_closed:
                close_debt = close_row['debt_before']
                close_ltv = close_row['ltv_before']
                close_borrow_rate = close_row['borrow_rate_before']
            else:
                close_debt = None
                close_ltv = None
                close_borrow_rate = None
            
            debt_values = position_events['debt_after'].values
            ltv_values = position_events['ltv_after'].values
            max_debt = np.max(debt_values) if len(debt_values) > 0 else open_debt
            max_ltv = np.max(ltv_values) if len(ltv_values) > 0 else open_ltv

            non_zero_mask = (debt_values > 0) & (ltv_values > 0)
            if non_zero_mask.any():
                first_valid_idx = np.argmax(non_zero_mask)
                open_debt = debt_values[first_valid_idx]
                open_ltv = ltv_values[first_valid_idx]
            else:
                open_debt = debt_values[0]
                open_ltv = ltv_values[0]
            
            position_events_ex_open = position_events.iloc[1:] if len(position_events) > 1 else pd.DataFrame()
            n_repays = (position_events_ex_open['type'] == 'MarketRepay').sum() if not position_events_ex_open.empty else 0
            n_borrows = (position_events_ex_open['type'] == 'MarketBorrow').sum() if not position_events_ex_open.empty else 0
            
            twenty_four_hours = 24 * 3600
            borrow_mask = (user_df['type'] == 'MarketBorrow') & \
                          (user_df['timestamp'] > open_ts) & \
                          (user_df['timestamp'] <= open_ts + twenty_four_hours)
            leverage_factor = borrow_mask.sum()
            
            positions.append({
                'user_address': user,
                'open_timestamp': open_ts,
                'open_datetime': open_row['datetime'],
                'open_debt': open_debt,
                'open_ltv': open_ltv,
                'open_borrow_rate': open_borrow_rate,
                'leverage_factor': leverage_factor,
                'close_timestamp': close_ts,
                'close_datetime': close_row['datetime'] if is_closed else None,
                'close_debt': close_debt,
                'close_ltv': close_ltv,
                'close_borrow_rate': close_borrow_rate,
                'max_debt': max_debt,
                'max_ltv': max_ltv,
                'n_repays': n_repays,
                'n_borrows': n_borrows,
                'is_closed': is_closed,
                'position_events': position_events,
                'user_events_df': user_df,
                'open_idx': open_idx,
                'close_idx': close_idx if is_closed else None
            })
    
    return positions

def compute_position_features(positions, market_events, market_hourly, market_spikes, suppliers_share, borrowers_share, market_name):
    if not positions:
        return pd.DataFrame()
    
    positions_df = pd.DataFrame([{
        'user_address': p['user_address'],
        'open_timestamp': p['open_timestamp'],
        'open_datetime': p['open_datetime'],
        'open_debt': p['open_debt'],
        'open_ltv': p['open_ltv'],
        'open_borrow_rate': p['open_borrow_rate'],
        'leverage_factor': p['leverage_factor'],
        'close_timestamp': p['close_timestamp'],
        'close_datetime': p['close_datetime'],
        'close_debt': p['close_debt'],
        'close_ltv': p['close_ltv'],
        'close_borrow_rate': p['close_borrow_rate'],
        'max_debt': p['max_debt'],
        'max_ltv': p['max_ltv'],
        'n_repays': p['n_repays'],
        'n_borrows': p['n_borrows'],
        'is_closed': p['is_closed'],
        'position_events': p['position_events'],
        'user_events_df': p['user_events_df'],
    } for p in positions])
    
    hourly_sorted = market_hourly.sort_values('timestamp')
    hourly_ts = hourly_sorted['timestamp'].values
    hourly_total_borrow = hourly_sorted['total_borrow'].values
    hourly_total_supply = hourly_sorted['total_supply'].values
    hourly_utilization = hourly_sorted['utilization'].values
    hourly_borrow_rate = hourly_sorted['borrow_rate'].values
    
    def get_hourly_at(ts):
        idx = np.searchsorted(hourly_ts, ts, side='right') - 1
        if idx < 0:
            return None, None, None, None
        return hourly_total_borrow[idx], hourly_total_supply[idx], hourly_utilization[idx], hourly_borrow_rate[idx]
    
    open_vals = [get_hourly_at(ts) for ts in positions_df['open_timestamp']]
    positions_df['total_borrow_open'] = [v[0] for v in open_vals]
    positions_df['total_supply_open'] = [v[1] for v in open_vals]
    positions_df['utilization_open'] = [v[2] for v in open_vals]
    positions_df['total_debt_open'] = positions_df['total_borrow_open']
    positions_df['total_liquidity_open'] = positions_df['total_supply_open'] - positions_df['total_borrow_open']
    positions_df['position_size_share_open'] = positions_df['open_debt'] / positions_df['total_borrow_open'].replace(0, np.nan)
    
    max_debt_ts_list = []
    for _, row in positions_df.iterrows():
        pos_events = row['position_events']
        if len(pos_events) > 0:
            max_debt_ts = pos_events[pos_events['debt_after'] == row['max_debt']]['timestamp'].iloc[0]
        else:
            max_debt_ts = row['open_timestamp']
        max_debt_ts_list.append(max_debt_ts)
    positions_df['max_debt_ts'] = max_debt_ts_list
    
    max_borrow_vals = [get_hourly_at(ts)[0] for ts in max_debt_ts_list]
    positions_df['total_borrow_max'] = max_borrow_vals
    positions_df['position_size_share_max'] = positions_df['max_debt'] / positions_df['total_borrow_max'].replace(0, np.nan)
    
    duration_hours = []
    time_to_first_action = []
    avg_time_between_actions = []
    max_time_between_actions = []
    n_actions_total = []
    avg_repay_ratio = []
    position_end_list = []
    
    for _, row in positions_df.iterrows():
        open_ts = row['open_timestamp']
        close_ts = row['close_timestamp'] if row['is_closed'] else None
        pos_events = row['position_events']
        
        if close_ts is None:
            last_event_ts = pos_events['timestamp'].max()
            duration_sec = last_event_ts - open_ts
            position_end = last_event_ts
        else:
            duration_sec = close_ts - open_ts
            position_end = close_ts
        duration_hours.append(duration_sec / 3600.0)
        position_end_list.append(position_end)
        
        actions = pos_events.iloc[1:] if len(pos_events) > 1 else pd.DataFrame()
        if not actions.empty:
            actions = actions[actions['type'].isin(['MarketBorrow', 'MarketRepay'])]
        n_act = len(actions)
        n_actions_total.append(n_act)
        
        if n_act > 0:
            act_ts = actions['timestamp'].values
            tfa = (act_ts[0] - open_ts) / 3600.0
            time_to_first_action.append(tfa)
            if n_act > 1:
                gaps = np.diff(act_ts) / 3600.0
                avg_time_between_actions.append(gaps.mean())
                max_time_between_actions.append(gaps.max())
            else:
                avg_time_between_actions.append(np.nan)
                max_time_between_actions.append(np.nan)
        else:
            time_to_first_action.append(np.nan)
            avg_time_between_actions.append(np.nan)
            max_time_between_actions.append(np.nan)
        
        repay_actions = pos_events[pos_events['type'] == 'MarketRepay']
        ratios = []
        for _, ra in repay_actions.iterrows():
            if ra['debt_before'] > 0:
                ratios.append(ra['assets_usd'] / ra['debt_before'])
        avg_repay_ratio.append(np.mean(ratios) if ratios else np.nan)
    
    positions_df['duration_hours'] = duration_hours
    positions_df['time_to_first_action'] = time_to_first_action
    positions_df['avg_time_between_actions'] = avg_time_between_actions
    positions_df['max_time_between_actions'] = max_time_between_actions
    positions_df['n_actions_total'] = n_actions_total
    positions_df['avg_repay_ratio'] = avg_repay_ratio
    positions_df['position_end'] = position_end_list
    
    if not market_spikes.empty:
        spike_starts = market_spikes['spike_trigger_timestamp'].values
        spike_ends = market_spikes['spike_recovery_timestamp'].values
        open_ts_arr = positions_df['open_timestamp'].values
        close_ts_arr = positions_df['close_timestamp'].values
        is_closed_arr = positions_df['is_closed'].values
        pos_end_arr = positions_df['position_end'].values
        
        was_active = np.zeros(len(positions_df), dtype=bool)
        num_spikes = np.zeros(len(positions_df), dtype=int)
        closed_during = np.zeros(len(positions_df), dtype=bool)
        
        for i in range(len(spike_starts)):
            s_start = spike_starts[i]
            s_end = spike_ends[i]
            overlap = (open_ts_arr <= s_end) & (pos_end_arr >= s_start)
            was_active |= overlap
            num_spikes += overlap
            
            if is_closed_arr.any():
                closed_mask = overlap & (close_ts_arr >= s_start) & (close_ts_arr <= s_end)
                closed_during |= closed_mask
        
        positions_df['was_active_during_spike'] = was_active
        positions_df['num_spikes_experienced'] = num_spikes
        positions_df['closed_during_spike'] = closed_during
    else:
        positions_df['was_active_during_spike'] = False
        positions_df['num_spikes_experienced'] = 0
        positions_df['closed_during_spike'] = False
    
    borrowers_market = borrowers_share[borrowers_share['market'] == market_name].sort_values('timestamp')
    borrowers_ts = borrowers_market['timestamp'].values
    borrowers_users = borrowers_market['user_address'].values
    borrowers_share_val = borrowers_market['share'].values
    
    def is_top_borrower(user, open_ts, close_ts, is_closed):
        idx_start = np.searchsorted(borrowers_ts, open_ts, side='left')
        if is_closed and close_ts is not None:
            idx_end = np.searchsorted(borrowers_ts, close_ts, side='right')
        else:
            idx_end = len(borrowers_ts)
        for i in range(idx_start, idx_end):
            if borrowers_users[i] == user and borrowers_share_val[i] > 0:
                return True
        return False
    
    debtors_rank_flags = []
    for _, row in positions_df.iterrows():
        debtors_rank_flags.append(
            is_top_borrower(row['user_address'], row['open_timestamp'], row['close_timestamp'], row['is_closed'])
        )
    positions_df['debtors_rank'] = debtors_rank_flags
    
    suppliers_market = suppliers_share[suppliers_share['market'] == market_name].sort_values('timestamp')
    suppliers_ts = suppliers_market['timestamp'].values
    suppliers_hhi = suppliers_market['hhi'].values
    
    def get_hhi_at(ts):
        idx = np.searchsorted(suppliers_ts, ts, side='right') - 1
        if idx < 0:
            return np.nan
        return suppliers_hhi[idx]
    
    positions_df['concentration_hhi_open'] = positions_df['open_timestamp'].apply(get_hhi_at)
    
    borrowers_market_open = borrowers_share[
        (borrowers_share['market'] == market_name) & 
        (borrowers_share['side'] == 'borrow') &
        (borrowers_share['user_address'] != 'other')
    ].sort_values(['timestamp', 'share'], ascending=[True, False])
    
    top3_map = {}
    for ts, group in borrowers_market_open.groupby('timestamp'):
        top3_map[ts] = group['share'].iloc[:3].sum()
    borrowers_open_ts_all = np.array(list(top3_map.keys()))
    borrowers_open_top3 = np.array(list(top3_map.values()))
    
    def get_top3_at(ts):
        idx = np.searchsorted(borrowers_open_ts_all, ts, side='right') - 1
        if idx < 0:
            return np.nan
        return borrowers_open_top3[idx]
    
    positions_df['top3_share_open'] = positions_df['open_timestamp'].apply(get_top3_at)
    
    avg_borrow_rates = []
    for _, row in positions_df.iterrows():
        mask = (hourly_ts >= row['open_timestamp']) & (hourly_ts <= row['position_end'])
        if mask.any():
            avg_borrow_rates.append(hourly_borrow_rate[mask].mean())
        else:
            avg_borrow_rates.append(row['open_borrow_rate'])
    positions_df['avg_borrow_rate_position'] = avg_borrow_rates
    
    result_cols = [
        'user_address', 'market', 'open_timestamp', 'open_datetime', 'open_debt', 'open_ltv',
        'open_borrow_rate', 'leverage_factor', 'close_timestamp', 'close_datetime', 'close_debt',
        'close_ltv', 'close_borrow_rate', 'max_debt', 'max_ltv', 'n_repays', 'n_borrows',
        'is_closed', 'duration_hours', 'time_to_first_action', 'avg_time_between_actions',
        'max_time_between_actions', 'n_actions_total', 'avg_repay_ratio', 'was_active_during_spike',
        'num_spikes_experienced', 'closed_during_spike', 'position_size_share_open',
        'position_size_share_max', 'debtors_rank', 'utilization_open', 'total_debt_open',
        'total_liquidity_open', 'concentration_hhi_open', 'top3_share_open',
        'avg_borrow_rate_position'
    ]
    
    positions_df['market'] = market_name
    result_df = positions_df[result_cols].sort_values('open_timestamp').reset_index(drop=True)
    result_df['position_index'] = result_df.index
    return result_df


def build_market_positions_dataset(market_name):
    events = load_market_events(market_name)
    hourly = load_market_hourly(market_name)
    spikes_all = load_spikes()
    spikes_all['spike_trigger_timestamp'] = spikes_all['spike_trigger_datetime'].astype(int) // 10**9
    spikes_all['spike_recovery_timestamp'] = spikes_all['spike_recovery_datetime'].astype(int) // 10**9
    market_spikes = spikes_all[spikes_all['market_name'] == market_name]
    suppliers_share = load_suppliers_share()
    borrowers_share = load_borrowers_share()
    
    positions_raw = extract_user_positions(events)
    positions_df = compute_position_features(
        positions_raw, events, hourly, market_spikes,
        suppliers_share, borrowers_share, market_name
    )
    return positions_df

def build_all_positions_dataset(market_list):
    all_positions = []
    for market in tqdm_notebook(market_list):
        try:
            market_positions = build_market_positions_dataset(market)
            parts = market.split('_')
            collateral_asset = parts[1] if len(parts) > 1 else None
            loan_asset = parts[2] if len(parts) > 2 else None
            market_positions["collateral_asset"] = collateral_asset
            market_positions["loan_asset"] = loan_asset

            all_positions.append(market_positions)
        except Exception as e:
            print(f"Data files for market {market} not found, skipping. Error: {e}")
            continue
    if all_positions:
        final_df = pd.concat(all_positions, ignore_index=True)
        final_df = final_df.sort_values(['market', 'open_timestamp']).reset_index(drop=True)
        return final_df
    else:
        return pd.DataFrame()


In [5]:
INTERESTING_MARKETS = CRYPTO_MARKETS
dataset = build_all_positions_dataset(INTERESTING_MARKETS)
dataset.head(3)
output_path = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/users_positions/crypto_tokens_positions.csv"
dataset.to_csv(output_path, index=False)
print(f"Dataset saved to {output_path}")

/var/folders/hj/pbs977kd43s6n1l9z3mxrj200000gn/T/ipykernel_39928/163822572.py:433: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for market in tqdm_notebook(market_list):


  0%|          | 0/8 [00:00<?, ?it/s]

Dataset saved to /Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/users_positions/crypto_tokens_positions.csv


In [6]:
dataset.head(40)
# print(dataset.shape)
# dataset.isna().sum()

,user_address,market,open_timestamp,open_datetime,open_debt,open_ltv,open_borrow_rate,leverage_factor,close_timestamp,close_datetime,close_debt,close_ltv,close_borrow_rate,max_debt,max_ltv,n_repays,n_borrows,is_closed,duration_hours,time_to_first_action,avg_time_between_actions,max_time_between_actions,n_actions_total,avg_repay_ratio,was_active_during_spike,num_spikes_experienced,closed_during_spike,position_size_share_open,position_size_share_max,debtors_rank,utilization_open,total_debt_open,total_liquidity_open,concentration_hhi_open,top3_share_open,avg_borrow_rate_position,position_index,collateral_asset,loan_asset
0,0xf8b9D83574574A345c32905a2A62D1d84650A0Ae,base_cbbtc_usdc_full,1726217705,2024-09-13 08:55:05,1.0,0.172022,0.041709,0,NaN,NaT,NaN,NaN,NaN,1.000000,0.172022,0,0,False,0.000000,NaN,NaN,NaN,0,NaN,False,0,False,NaN,NaN,False,0.000000,0.000000,0.000000,NaN,NaN,0.041709,0,cbbtc,usdc
1,0xf8b9D83574574A345c32905a2A62D1d84650A0Ae,base_cbbtc_usdc_full,1726217705,2024-09-13 08:55:05,1.0,0.172022,0.041709,0,NaN,NaT,NaN,NaN,NaN,1.000000,0.172022,0,0,False,0.000000,NaN,NaN,NaN,0,NaN,False,0,False,NaN,NaN,False,0.000000,0.000000,0.000000,NaN,NaN,0.041709,1,cbbtc,usdc
2,0x9aE9A52Eb952aE82c592e109f04C844B94e9975a,base_cbbtc_usdc_full,1726241775,2024-09-13 15:36:15,150000.0,0.700799,0.041616,0,1.727133e+09,2024-09-23 23:02:29,-8.357457,-0.231041,0.026968,150000.000000,0.700799,3,0,True,247.437222,12.058889,98.025000,195.958889,3,1.000004,False,0,False,150000.000000,150000.000000,False,0.983789,1.000000,0.016478,NaN,NaN,0.030925,2,cbbtc,usdc
3,0x9aE9A52Eb952aE82c592e109f04C844B94e9975a,base_cbbtc_usdc_full,1726241775,2024-09-13 15:36:15,150000.0,0.700799,0.041616,0,1.727133e+09,2024-09-23 23:02:29,-8.357457,-0.231041,0.026968,150000.000000,0.700799,3,0,True,247.437222,12.058889,98.025000,195.958889,3,1.000004,False,0,False,150000.000000,150000.000000,False,0.983789,1.000000,0.016478,NaN,NaN,0.030925,3,cbbtc,usdc
4,0x4523b791292da89A9194B61bA4CD9d98f2af68E0,base_cbbtc_usdc_full,1726243993,2024-09-13 16:13:13,150.0,0.640749,0.041747,0,1.729762e+09,2024-10-24 09:30:37,448.978001,0.549055,0.014769,3411.000000,0.695630,2,3,True,977.290000,303.335000,168.488750,325.871667,5,0.938548,False,0,False,0.001000,0.000813,False,0.980745,150001.000033,2944.899679,NaN,NaN,0.022854,4,cbbtc,usdc
5,0x4523b791292da89A9194B61bA4CD9d98f2af68E0,base_cbbtc_usdc_full,1726243993,2024-09-13 16:13:13,150.0,0.640749,0.041747,0,1.729762e+09,2024-10-24 09:30:37,448.978001,0.549055,0.014769,3411.000000,0.695630,2,4,True,977.290000,0.000000,195.458000,325.871667,6,0.938548,False,0,False,0.001000,0.000813,False,0.980745,150001.000033,2944.899679,NaN,NaN,0.022854,5,cbbtc,usdc
6,0x7798Ba9512B5A684C12e31518923Ea4221A41Fb9,base_cbbtc_usdc_full,1726248019,2024-09-13 17:20:19,250.0,0.491057,0.041940,1,1.737410e+09,2025-01-20 21:54:25,250.000000,0.284800,0.070449,250.000000,0.491057,1,1,True,3100.568333,0.022222,3100.546111,3100.546111,2,1.021059,False,0,False,0.001665,0.001665,False,0.980566,150151.441154,2975.845278,NaN,NaN,0.058766,6,cbbtc,usdc
7,0x7798Ba9512B5A684C12e31518923Ea4221A41Fb9,base_cbbtc_usdc_full,1726248099,2024-09-13 17:21:39,250.0,0.491057,0.042474,0,1.737410e+09,2025-01-20 21:54:25,250.000000,0.284800,0.070449,250.000000,0.491057,1,0,True,3100.546111,3100.546111,NaN,NaN,1,1.021059,False,0,False,0.001665,0.001665,False,0.980566,150151.441154,2975.845278,NaN,NaN,0.058766,7,cbbtc,usdc
8,0x17316bdCB2734D87b2c0291eE59fF02476B18bbB,base_cbbtc_usdc_full,1726264953,2024-09-13 22:02:33,17651.0,0.500612,0.039947,0,1.726361e+09,2024-09-15 00:42:29,17651.000000,0.502046,0.042246,17651.000000,0.500612,1,0,True,26.665556,26.665556,NaN,NaN,1,1.000105,False,0,False,0.117356,0.117356,False,0.971706,150405.334892,4379.512757,NaN,NaN,0.033873,8,cbbtc,usdc
9,0x17316bdCB2734D87b2c0291eE59fF02476B18bbB,base_cbbtc_usdc_full,1726264953,2024-09-13 22:02:33,17651.0,0.500612,0.039947,0,1.726361e+09,2024-09-15 00:42:29,17651.000000,0.502046,0.042246,17651.000000,0.500612,1,0,Tr